In [ ]:
import subprocess
subprocess.run([
    'bash', '-c',
    'echo /usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib > /etc/ld.so.conf.d/cuda13.conf && ldconfig'
], capture_output=True)

import os
os.environ['LD_LIBRARY_PATH'] = (
    '/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:' +
    os.environ.get('LD_LIBRARY_PATH', '')
)
print("Library path configured.")

In [ ]:
!pip install -q --upgrade --force-reinstall \
    transformers==4.44.0 \
    peft==0.12.0 \
    bitsandbytes \
    accelerate==0.34.0 \
    gradio

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import subprocess
subprocess.run([
    'bash', '-c',
    'echo /usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib > /etc/ld.so.conf.d/cuda13.conf && ldconfig'
], capture_output=True)

import os
os.environ['LD_LIBRARY_PATH'] = (
    '/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:' +
    os.environ.get('LD_LIBRARY_PATH', '')
)
print("Library path configured.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_MODEL = "SafiaSidimoussa/medical-chatbot-tinyllama"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_MODEL)
model.eval()
print("Model loaded.")

In [ ]:
def ask(question, max_new_tokens=250):
    prompt = f"""### Instruction:
{question}

### Response:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.4,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("### Response:")[-1].strip()

    # Cut off at last complete sentence
    sentences = response.split(". ")
    if len(sentences) > 1:
        response = ". ".join(sentences[:-1]) + "."

    return response

print("ask() function ready.")

In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft(), css="""
    .gradio-container { max-width: 800px; margin: auto; }
    footer { display: none !important; }
""") as demo:
    gr.Markdown("""
    # 🏥 Medical Symptom Chatbot
    *Fine-tuned on 1,000 medical Q&A examples using QLoRA*

    > ⚠️ **Disclaimer:** This chatbot provides general medical information only.  
    > Always consult a qualified healthcare provider for medical concerns.
    """)

    chatbot = gr.Chatbot(
        height=450,
        bubble_full_width=False,
        show_label=False,
        type="messages",
    )

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask a medical question...",
            scale=9,
            show_label=False,
            container=False,
        )
        send = gr.Button("Send", scale=1, variant="primary")

    with gr.Row():
        gr.Examples(
            examples=[
                "What are the symptoms of diabetes?",
                "What are the warning signs of a heart attack?",
                "I have a fever and sore throat. What could it be?",
                "What are the early signs of a stroke?",
            ],
            inputs=msg,
            label="Try these examples:"
        )

    def chat(message, history):
        if not message.strip():
            return "", history
        response = ask(message)
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": response})
        return "", history

    msg.submit(chat, [msg, chatbot], [msg, chatbot])
    send.click(chat, [msg, chatbot], [msg, chatbot])

demo.launch(share=True)